In [3]:
import json
from tqdm import tqdm
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv
load_dotenv()

JSON_PATH = "dataset/cddm_knowledge_graph_triplets.json"   

driver = GraphDatabase.driver(os.getenv("NEO4J_URI"), auth=(os.getenv("NEO4J_USER"), os.getenv("NEO4J_PASSWORD")))

with open(JSON_PATH, "r") as f:
    data = json.load(f)

with driver.session() as session:
    for source, relation, target in tqdm(data, desc="Creating graph", unit="rel"):
        relation = relation.upper().replace(" ", "_")
        query = f"""
        MERGE (a:Entity {{name: $source}})
        MERGE (b:Entity {{name: $target}})
        MERGE (a)-[:{relation}]->(b)
        """
        session.run(query, source=source, target=target)

print("Graph built!")
driver.close()

Creating graph: 100%|██████████| 1001/1001 [00:05<00:00, 179.56rel/s]

Graph built!
